# VDA-HyperQuant: Colab Evaluation
This notebook demonstrates how to load Video-Depth-Anything (VDA), apply the **VDA-HyperQuant** model surgery, and run evaluation on sample video frames.

### Step 1: Clone the Repository and Install Dependencies

In [1]:
# Clone the vdaquant repo (skip if already present)
import os
if not os.path.exists('vdaquant'):
    !git clone https://github.com/anrd30/vdaquant.git
%cd vdaquant

# Install dependencies (VDA + HyperQuant)
!pip install -q torch torchvision numpy opencv-python pillow matplotlib timm einops easydict imageio imageio-ffmpeg tqdm decord

Cloning into 'vdaquant'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 48 (delta 11), reused 47 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 29.77 KiB | 9.92 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/vdaquant


### Step 2: Clone Video-Depth-Anything

In [2]:
# Clone the original VDA repo inside our workspace
!git clone https://github.com/DepthAnything/Video-Depth-Anything.git

Cloning into 'Video-Depth-Anything'...
remote: Enumerating objects: 356, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 356 (delta 143), reused 82 (delta 82), pack-reused 177 (from 1)
Receiving objects: 100% (356/356), 7.60 MiB | 14.54 MiB/s, done.
Resolving deltas: 100% (197/197), done.


### Step 3: Run the Verification Tests
Make sure all the mathematical core modules (Hadamard butterfly, D4 lattice, QJL) are working perfectly on the Colab environment.

In [3]:
!python3 tests/test_core.py

  VDA-HyperQuant — Core Verification Tests

[1] Hadamard Orthogonality
  ✓ Hadamard orthogonality d=4: max error = 0.00e+00
  ✓ Hadamard orthogonality d=8: max error = 0.00e+00
  ✓ Hadamard orthogonality d=16: max error = 0.00e+00
  ✓ Hadamard orthogonality d=64: max error = 0.00e+00
  ✓ Hadamard orthogonality d=256: max error = 0.00e+00

[2] Fast Hadamard Transform vs Matrix
  ✓ FHT matches matrix multiply (d=64): max error = 7.15e-07

[3] RHT Invertibility
  ✓ RHT invertibility (64-dim, power-of-2): max error = 4.77e-07
  ✓ RHT invertibility (384→512→384): max error = 5.36e-07

[4] RHT Outlier Suppression (THE KEY EXPERIMENT)
  ✓ Outlier suppression: before max/mean = 37.7, after = 1.7, reduction = 22.8x

[5] D4 Lattice Quantizer
  ✓ D4 lattice quantizer: output shape torch.Size([100, 64]), method=lattice_d4, effective_bits=3.75

[6] Quantizer MSE Comparison

  Quantizer MSE Comparison @ 4-bit:
  Method                           MSE
  -----------------------------------
  Scalar (NO 

### Step 4: Run End-to-End Evaluation
This runs the full evaluation on the real Video-Depth-Anything model: downloading the ViT-Small checkpoint, running the FP32 baseline, applying model surgery, and comparing accuracy, speed (FPS), and depth map outputs.

In [4]:
# Quick smoke test (5 frames). For full Sintel eval on GPU, use run_pareto_benchmark_suite.py instead.
!python3 scripts/run_vda_quant_eval.py --bits 4 --quantizer lattice_d4 --use_qjl --num_frames 5

xFormers not available
xFormers not available
xFormers not available
xFormers not available
Using device: cuda
--2026-07-01 11:56:33--  https://huggingface.co/depth-anything/Video-Depth-Anything-ViT-Small/resolve/main/video_depth_anything_vits.pth
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.118, 18.164.174.55, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 401 Unauthorized

Username/Password Authentication Failed.
--2026-07-01 11:56:33--  https://raw.githubusercontent.com/intel-isl/DensePredictionTransformer/master/dpt/assets/dog.jpg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-07-01 11:56:34 ERROR 404: Not Found.


[1/4] Loading original FP32 model.

### Step 5: Visualize the Results
Display the comparison between original FP32 depth maps and quantized depth maps.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

comp_path = 'outputs/depth_comparison.png'
if os.path.exists(comp_path):
    plt.figure(figsize=(15, 5))
    plt.imshow(Image.open(comp_path))
    plt.axis('off')
    plt.show()
else:
    print("Run the evaluation cell in Step 4 first!")